# Phase 05: Manifold Topology and Visual Metrics

This notebook performs topological analysis on the Phase 04 production telemetry, including 3D phase-space reconstruction, Poincaré sections, and Correlation Dimension ($D_2$) estimation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial import KDTree
import mmap
import struct

# 1. Ingestion and Dynamic Parameter Calculation
def ingest_telemetry(file_path):
    with open(file_path, "rb") as f:
        mmapped_file = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ)
        data_start = 128
        block_size = 12
        raw_data = np.frombuffer(mmapped_file, dtype=np.float32, offset=data_start, count=((mmapped_file.size() - data_start) // block_size) * 3)
        raw_data = raw_data.reshape(-1, 3).astype(np.float64)
    return raw_data

def calculate_mutual_information(ts, max_lag=50):
    n = len(ts)
    bins = 20
    ts_binned = np.digitize(ts, np.linspace(np.min(ts), np.max(ts), bins))
    mi = []
    for lag in range(1, max_lag):
        x = ts_binned[:-lag]
        y = ts_binned[lag:]
        hist_xy = np.histogram2d(x, y, bins=bins)[0]
        p_xy = hist_xy / np.sum(hist_xy)
        p_x = np.sum(p_xy, axis=1)
        p_y = np.sum(p_xy, axis=0)
        mi_val = 0
        for i in range(bins):
            for j in range(bins):
                if p_xy[i, j] > 0:
                    mi_val += p_xy[i, j] * np.log2(p_xy[i, j] / (p_x[i] * p_y[j]))
        mi.append(mi_val)
    mi = np.array(mi)
    for i in range(1, len(mi)-1):
        if mi[i] < mi[i-1] and mi[i] < mi[i+1]:
            return i + 1
    return 1

data = ingest_telemetry("../04-production-execution-stability-profiling/production_results.bin")
ts = data[:, 0]
vorticity = data[:, 1]

# Dynamic tau using AMI
tau = calculate_mutual_information(ts)
m = 3 # Embedding dimension

X = np.array([[ts[i+j*tau] for j in range(m)] for i in range(len(ts)-m*tau)])
vort_mapped = vorticity[:len(X)]


In [ ]:
# 2. 3D Phase-Space Attractor Projection with Physical Mapping
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X[::100, 0], X[::100, 1], X[::100, 2], s=0.5, c=vort_mapped[::100], cmap='inferno')
ax.set_xlabel('x(t)')
ax.set_ylabel('x(t+τ)')
ax.set_zlabel('x(t+2τ)')
plt.title('3D Attractor Reconstruction (Colored by Vorticity)')
plt.savefig('attractor_projection.pdf')
plt.show()

In [ ]:
# 3. Poincaré Section (Adaptive Hyperplane at Mean)
mean_plane = np.mean(X[:, 2])
cut = X[(X[:, 2][:-1] < mean_plane) & (X[:, 2][1:] >= mean_plane)]
plt.scatter(cut[:, 0], cut[:, 1], s=1)
plt.title(f'Poincaré Section (Plane z={mean_plane:.3f})')
plt.savefig('poincare_section.pdf')
plt.show()

In [ ]:
# 4. Correlation Dimension (D2) with Dynamic Range
sample_X = X[::100]
tree = KDTree(sample_X)
r_min = np.min(tree.query(sample_X, k=2)[0][:,1])
r_max = np.max(np.std(sample_X, axis=0))
r_vals = np.logspace(np.log10(r_min), np.log10(r_max), 20)
C = [tree.count_neighbors(tree, r=r) / len(sample_X)**2 for r in r_vals]
plt.loglog(r_vals, C, 'o-')
plt.xlabel('log r')
plt.ylabel('log C(r)')
plt.title('Correlation Dimension D2 Estimation')
plt.savefig('correlation_dimension.pdf')
plt.show()